# Member 3 — Leakage and Validation

**Project:** Hotel Booking Demand — Cancellation Risk  
**Owner:** IT24103815 (Member 3)

Purpose: Check for data leakage and establish validation strategy.

## Cell 1 — Load validated data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

DATA_PATH = Path("../../data/processed/hotel_bookings_validated_phase3.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("data/processed/hotel_bookings_validated_phase3.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Cancellation rate:", f"{df['is_canceled'].mean():.2%}")


Dataset shape: (119210, 32)
Cancellation rate: 37.08%


## Cell 2 — Select initial model features

In [2]:

features = [
    "hotel",
    "lead_time",
    "arrival_date_year",
    "arrival_date_month",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "meal",
    "reserved_room_type",
    "country",
    "market_segment",
    "distribution_channel",
    "customer_type",
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "days_in_waiting_list"
]

X = df[features].copy()
y = df["is_canceled"].astype(int)

excluded = [
    col for col in df.columns
    if col not in features + ["is_canceled"]
]

print("Selected features:", X.shape[1])
print("Excluded or deferred features:", excluded)
print("Missing values in selected features:", X.isna().sum().sum())


Selected features: 21
Excluded or deferred features: ['assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']
Missing values in selected features: 0


## Cell 3 — Split training and testing data

In [3]:
groups = pd.util.hash_pandas_object(X, index=False)

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]

print("Training:", X_train.shape)
print("Testing:", X_test.shape)
print("Train cancellation rate:", f"{y_train.mean():.2%}")
print("Test cancellation rate:", f"{y_test.mean():.2%}")

assert set(groups.iloc[train_idx]).isdisjoint(
    set(groups.iloc[test_idx])
)

print("No identical feature groups across train and test.")


Training: (95334, 21)
Testing: (23876, 21)
Train cancellation rate: 37.01%
Test cancellation rate: 37.35%
No identical feature groups across train and test.


## Cell 4 — Prepare cross-validation

In [4]:

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

folds = list(
    cv.split(X_train, y_train, groups=groups_train)
)

print("Cross-validation folds:", len(folds))

for i, (tr, val) in enumerate(folds, start=1):
    rate = y_train.iloc[val].mean()

    print(
        f"Fold {i}:",
        f"{len(tr)} training,",
        f"{len(val)} validation,",
        f"Cancellation rate: {rate:.2%}"
    )


Cross-validation folds: 5
Fold 1: 76286 training, 19048 validation, Cancellation rate: 37.23%
Fold 2: 76083 training, 19251 validation, Cancellation rate: 35.72%
Fold 3: 76377 training, 18957 validation, Cancellation rate: 38.25%
Fold 4: 76270 training, 19064 validation, Cancellation rate: 36.86%
Fold 5: 76320 training, 19014 validation, Cancellation rate: 37.00%


### Cross-Validation Results

Five-fold Stratified Group Cross-Validation was used to maintain similar cancellation distributions while keeping identical feature combinations within the same fold.

The validation cancellation rates ranged from 35.72% to 38.25%, which is reasonably close to the overall training cancellation rate of 37.01%.

The results indicate an acceptable class distribution across the validation folds. This strategy will be used consistently during model training and comparison.

## Cell 5 — Save the split indices

In [5]:
output = (
    DATA_PATH.parent.parent.parent /
    "results/splits/group_holdout_indices.npz"
)

output.parent.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    output,
    train_idx=train_idx,
    test_idx=test_idx
)

print("Split indices saved:", output)


Split indices saved: ..\..\results\splits\group_holdout_indices.npz


## Feature Selection and Validation Summary

The initial prediction model uses 21 booking-related features selected according to their expected availability at reservation confirmation. Variables containing possible future information or final booking outcomes are excluded or deferred.

The dataset is divided into approximately 80% training and 20% testing data using Stratified Group K-Fold. Records with identical selected feature values are kept within the same group to prevent overlap between training and testing.

Five-fold group-aware cross-validation is used for model development. All classifiers will follow the same feature set, train/test split, validation approach and evaluation metrics.

Preprocessing transformations will be fitted on training data only. The test set will be reserved for final evaluation after model selection.